## Section 1: Importing the Libraries

In [ ]:
import time
import torch
#used to create the generator and discriminator
from torch import nn 
# used to create visualisations
import matplotlib.pyplot as plt 
# used to set size of visualisations
from matplotlib import rcParams 
#used to clear the visualisations at the end of each epoch
from IPython import display 
#saving outputs to the notebook
%matplotlib inline

## Creating the Data

The generated dataset approximates a linear relationship between two variables X and Y using the formula
$$Y = XA + b$$
which is simply a vectorized version of linear regression

#### Creating the dataset

In [ ]:
X = torch.normal(0.0, 1, (1000, 2))
A = torch.tensor([[1, 2], [-0.1, 0.5]])
b = torch.tensor([1, 2])
data = torch.matmul(X, A) + b

#### Display dataset

In [ ]:
plt.scatter(data[:100, 0].detach().numpy(), data[:100, 1].detach().numpy())

#### Creating Dataset Iterateables

In [ ]:
batch_size = 8
dataset = torch.utils.data.TensorDataset(data)
data_iter = torch.utils.data.DataLoader(dataset, batch_size, shuffle=True)


## Section 2: The Model
### Generators and Discriminators
The generator needs to be trained to generator the discriminator cannot classify as fake. Sine the dataset is created using linear rigression, we use a single layered neural network.

![alt text](images/generator.png "Generator")

The discriminator is a slightly more complex neural network since it has to learn to classify the generated examples as real or fake.

![alt text](images/discriminator.png "Discriminator")

#### Generator Neural Net

In [ ]:
nnet_Gen = nn.Sequential(nn.Linear(2, 2))

#### Discriminator Neural Net

In [ ]:
nnet_Disc = nn.Sequential(
  nn.Linear(2, 5), nn.Tanh(),
  nn.Linear(5, 3), nn.Tanh(),
  nn.Linear(3, 1))

### Discriminator Updates

we use the real and synthetic batches to compute the loss by computing the loss on the discriminator outputs for real batch with a tensor of ones which denote real samples, and for synthetic batch with a tensor of zeroes which denote fakes. The final loss is an average of the two, since the discriminator's job is to correctly classify the two distributions as separate.

In [ ]:
def update_D(X, Z, nnet_D, nnet_G, loss, trainer_D):
    batch_size = X.shape[0]
    ones = torch.ones((batch_size,), device=X.device)
    zeros = torch.zeros((batch_size,), device=X.device)
    trainer_D.zero_grad()
    real_Y = nnet_D(X)
    synth_X = nnet_G(Z)
    synth_Y = nnet_D(synth_X.detach())
    loss_D = (loss(real_Y, ones.reshape(real_Y.shape)) +
              loss(synth_Y, zeros.reshape(synth_Y.shape))) / 2
    loss_D.backward()
    trainer_D.step()
    return loss_D